# Planners Comparison Study

This notebook demonstrates how to conduct a comprehensive comparison of different POMDP planning algorithms using the SimulationsAPI. We'll compare POMCPOW and PFT-DPW planners on Push POMDP and Light-Dark POMDP environments, showcasing how to evaluate algorithm performance across different problem domains.

## Overview

**Planning Algorithms Tested:**
- **POMCPOW**: Monte Carlo Tree Search with double progressive widening
- **PFT-DPW**: Progressive Function Transfer with Double Progressive Widening

**Environments Tested:**
- **Push POMDP**: Object manipulation with continuous actions
- **Light-Dark POMDP**: Navigation with position-dependent observation noise

**Key Features Demonstrated:**
- Environment configuration using EnvironmentConfigsAPI
- Pre-built action samplers from utils module for different action spaces
- Statistical analysis with confidence intervals
- Multi-environment, multi-algorithm evaluation
- Performance profiling and result visualization

## Setup and Imports

In [1]:
import numpy as np
from pathlib import Path

# Core POMDPPlanners imports
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.planners.mcts_planners.pomcpow import POMCPOW
from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.simulations.simulations_api import SimulationsAPI
from POMDPPlanners.core.simulation import EnvironmentRunParams
from POMDPPlanners.utils.action_samplers import UnitCircleActionSampler, DiscreteActionSampler

## Environment Setup

Setting up the environments using an the environment's API

In [2]:
env_config = EnvironmentConfigsAPI(discount_factor=0.95, debug=False)

push_env, push_belief = env_config.push_pomdp_config(n_particles=20)  # Reduced from 1000
print(f"Created Push POMDP environment: {push_env.__class__.__name__}")

light_dark_env, light_dark_belief = env_config.continuous_observations_discrete_actions_light_dark_pomdp_config(
    n_particles=20  # Reduced from 1000
)
print(f"Created Light-Dark POMDP environment: {light_dark_env.__class__.__name__}")

2025-09-29 19:07:26,327 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:240 - Initializing PushPOMDP environment with discount factor 0.95
2025-09-29 19:07:26,328 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:240 - Initializing ContinuousLightDarkPOMDPDiscreteActions environment with discount factor 0.95


Created Push POMDP environment: PushPOMDP
Created Light-Dark POMDP environment: ContinuousLightDarkPOMDPDiscreteActions


## Action Sampler Configuration

In [3]:
# Create action samplers based on available environments

# Push POMDP uses discrete actions: ["up", "down", "right", "left"]
push_action_sampler = DiscreteActionSampler(actions=push_env.get_actions())
print("Created discrete action sampler for Push POMDP")

# Light-Dark POMDP uses discrete actions: [0, 1, 2, 3]
light_dark_action_sampler = DiscreteActionSampler(actions=light_dark_env.get_actions())
print("Created discrete action sampler for Light-Dark POMDP")
## Planner Configuration

Created discrete action sampler for Push POMDP
Created discrete action sampler for Light-Dark POMDP


## Planner Configuration

Setting up POMCPOW and PFT-DPW planners for each environment:

In [4]:
# Configure planners for Push POMDP (or fallback environment)
# Reduced simulation counts for faster testing
push_planners = [
    POMCPOW(
        environment=push_env,
        discount_factor=0.95,
        depth=10,              # Reduced from 15
        exploration_constant=1.41,
        k_o=10,
        k_a=4,
        alpha_o=0.01,
        alpha_a=0.01,
        action_sampler=push_action_sampler,
        n_simulations=2000,     # Reduced from 1000
        name="POMCPOW_Push"
    ),
    PFT_DPW(
        environment=push_env,
        discount_factor=0.95,
        depth=10,              # Reduced from 15
        name="PFT_DPW_Push",
        action_sampler=push_action_sampler,
        k_a=10,
        alpha_a=0.01,
        k_o=10,
        alpha_o=0.01,
        exploration_constant=1.0,
        n_simulations=2000      # Reduced from 1000
    )
]
print(f"Created {len(push_planners)} planners for Push environment")

2025-09-29 19:07:30,556 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: POMCPOW_Push (debug=False)
2025-09-29 19:07:30,557 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: PFT_DPW_Push (debug=False)


Created 2 planners for Push environment


In [5]:
# Configure planners for Light-Dark POMDP (or fallback environment)
light_dark_planners = [
    POMCPOW(
        environment=light_dark_env,
        discount_factor=0.95,
        depth=10,              # Reduced from 20
        exploration_constant=100.,
        k_o=10,
        k_a=4,
        alpha_o=0.01,
        alpha_a=0.01,
        action_sampler=light_dark_action_sampler,
        n_simulations=2000,     # Reduced from 1500
        name="POMCPOW_LightDark"
    ),
    PFT_DPW(
        environment=light_dark_env,
        discount_factor=0.95,
        depth=10,              # Reduced from 20
        name="PFT_DPW_LightDark",
        action_sampler=light_dark_action_sampler,
        k_a=4,
        alpha_a=0.01,
        k_o=10,
        alpha_o=0.01,
        exploration_constant=100.,
        n_simulations=2000      # Reduced from 1500
    )
]
print(f"Created {len(light_dark_planners)} planners for Light-Dark environment")

2025-09-29 19:07:31,114 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: POMCPOW_LightDark (debug=False)
2025-09-29 19:07:31,115 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: PFT_DPW_LightDark (debug=False)


Created 2 planners for Light-Dark environment


## Simulation Configuration

In [6]:
# Create simulation configurations with reduced parameters for testing
environment_run_params = [
    EnvironmentRunParams(
        environment=push_env,
        belief=push_belief,
        policies=push_planners,
        num_episodes=10,       # Reduced from 100
        num_steps=15           # Reduced from 30
    ),
    EnvironmentRunParams(
        environment=light_dark_env,
        belief=light_dark_belief,
        policies=light_dark_planners,
        num_episodes=10,       # Reduced from 150
        num_steps=15           # Reduced from 25
    )
]

print(f"Created {len(environment_run_params)} environment run configurations")
for i, config in enumerate(environment_run_params):
    env_name = config.environment.__class__.__name__
    policies_names = [p.name for p in config.policies]
    print(f"  Config {i+1}: {env_name} with {len(config.policies)} policies")
    print(f"    Policies: {policies_names}")
    print(f"    Episodes: {config.num_episodes}, Steps: {config.num_steps}")

Created 2 environment run configurations
  Config 1: PushPOMDP with 2 policies
    Policies: ['POMCPOW_Push', 'PFT_DPW_Push']
    Episodes: 10, Steps: 15
  Config 2: ContinuousLightDarkPOMDPDiscreteActions with 2 policies
    Policies: ['POMCPOW_LightDark', 'PFT_DPW_LightDark']
    Episodes: 10, Steps: 15


## Running the Comparison Study

In [ ]:
# Initialize SimulationsAPI
api = SimulationsAPI(cache_dir_path=Path("./planners_comparison_results"), debug=True)

cache_dir_path = Path("./planners_comparison_results")

print("Starting planners comparison study...")
print(f"Running {len(environment_run_params)} environment configurations")

# Run simulation with statistical analysis
results, statistics_df = api.run_multiple_environments_and_policies_local_run_with_initial_debug_run(
    environment_run_params=environment_run_params,
    alpha=0.05,
    confidence_interval_level=0.95,
    experiment_name="Planners_Comparison_Study",
    n_jobs=-1,              # Use 1 core for testing
    enable_profiling=False,  # Disable profiling for simpler output
    cache_dir_path=cache_dir_path,
)

print("\nComparison study completed successfully!")
print(f"Results available for {len(results)} environments")
if statistics_df is not None:
    print(f"Statistics computed for {len(statistics_df)} configurations")
else:
    print("No statistics available")

2025-09-29 19:09:41,357 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: planners_comparison_results/logs/simulations_api_20250929_190941.log
2025-09-29 19:09:41,357 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:93 - Initialized SimulationsAPI
2025-09-29 19:09:41,357 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:441 - Starting simulation run with initial debug run
2025-09-29 19:09:41,358 - DEBUG: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:442 - Parameters: alpha=0.05, confidence_interval=0.95, n_jobs=-1
2025-09-29 19:09:41,358 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:457 - Created debug configurations with 2 environments
2025-09-29 19:09:41,358 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:46

Starting planners comparison study...
Running 2 environment configurations


Running tasks: 100%|██████████| 8/8 [00:00<00:00, 18558.87it/s]
2025-09-29 19:09:56,041 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: planners_comparison_results/logs/task_manager_20250929_190956.log
2025-09-29 19:09:56,042 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/task_managers.py:296 - Parallel processing completed in 14.62s
2025-09-29 19:09:56,042 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: planners_comparison_results/logs/task_manager_20250929_190956.log
2025-09-29 19:09:56,042 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/task_managers.py:297 - Results: 8 successful, 0 failed
2025-09-29 19:09:56,043 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: planners_comparison_results/logs/task_manager_20250929_1909

## Results Analysis and Visualization

### Accessing Evaluation Results

The simulation results are automatically saved and can be explored through MLflow's web interface. Run the following commands in your terminal:

```bash
# Navigate to the cache directory and launch MLflow UI
cd ./planners_comparison_results
mlflow ui --backend-store-uri ./mlruns
```

### Results Organization Structure

The evaluation results follow a hierarchical organization:

- **Environment Level**: Results are grouped by environment type (Push POMDP, Light-Dark POMDP)
- **Planner Comparisons**: Within each environment, performance metrics compare all tested planners
- **Statistical Analysis**: Confidence intervals and significance tests validate performance differences
- **Episode Tracking**: Individual episode results provide detailed performance insights

### Performance Metrics and Analysis

**Key Performance Metrics Available:**
- **Average Return**: Mean cumulative reward across episodes
- **Confidence Intervals**: Statistical bounds on performance estimates (95% confidence level)
- **Standard Deviation**: Measure of performance variability across episodes
- **Planning Time**: Computational efficiency and resource usage
- **Success Rates**: Episode completion and convergence statistics

## Summary

This notebook demonstrated:

1. **Multi-algorithm comparison** using POMCPOW and PFT-DPW planners
2. **Multi-environment evaluation** across different POMDP domains
3. **Statistical analysis** with confidence intervals and significance testing

The POMDPPlanners framework provides powerful tools for conducting rigorous algorithm comparisons across different problem domains, enabling both statistical rigor and practical insights for algorithm selection and tuning.

**Key Benefits:**
- Standardized evaluation protocols
- Flexible configuration options
- Performance profiling capabilities
- Reproducible experimental workflows

**Next Steps:**
- Hyper-parameter notebook